# Notebook 05 - Image Prompting and Visual Reasoning

This notebook focuses on image-first prompting patterns for charts, dashboards, screenshots, and other visual reasoning tasks.


## What you will learn

- how prompt wording changes image-grounded answers
- how to structure visual reasoning tasks around explicit evidence requests
- how to turn screenshots and charts into auditable outputs


In [ ]:
!pip install -q "transformers>=4.57.0" accelerate torch numpy pandas matplotlib Pillow
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "05_image_prompting_and_visual_reasoning"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)

from PIL import Image, ImageDraw

print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Create a synthetic chart task

Synthetic visuals make it easy to reason about the prompt contract before spending time on real datasets.


In [ ]:
canvas = Image.new("RGB", (640, 360), "white")
draw = ImageDraw.Draw(canvas)
values = [80, 120, 150, 220]
labels = ["Q1", "Q2", "Q3", "Q4"]
for idx, value in enumerate(values):
    x0 = 80 + idx * 120
    y0 = 300 - value
    draw.rectangle([x0, y0, x0 + 70, 300], fill="#4c78a8")
    draw.text((x0 + 18, 310), labels[idx], fill="black")
chart_path = NOTEBOOK_ROOT / "toy_chart.png"
canvas.save(chart_path)
plt.figure(figsize=(8, 4))
plt.imshow(canvas)
plt.axis("off")
plt.show()
print("Saved chart to", chart_path)


## Step 2: Compare prompt contracts

Prompts that require evidence and structure often behave better than prompts that ask for a vague free-form summary.


In [ ]:
prompt_styles = pd.DataFrame([
    {"prompt_style": "open summary", "prompt": "Describe the chart."},
    {"prompt_style": "evidence-first", "prompt": "List the highest and lowest bars, then explain the trend."},
    {"prompt_style": "structured", "prompt": "Return JSON with highest_period, lowest_period, and trend_summary."},
])
prompt_styles


## Step 3: Score the answer shape

A useful rubric checks answer correctness, evidence quality, and output format separately.


In [ ]:
rubric = pd.DataFrame([
    {"dimension": "correctness", "question": "Does the answer identify Q4 as highest?"},
    {"dimension": "grounding", "question": "Does the answer cite bars or labels rather than inventing causes?"},
    {"dimension": "format", "question": "Does the output follow the requested contract?"},
])
rubric


## Exercises and extensions

1. Swap the toy chart for a screenshot or dashboard tile from your own environment.
1. Write two failure prompts that are likely to cause overconfident but weakly grounded answers.
